# 04 — Kernel-informed structured pooling

Dilation-aware pooling guided by MultiROCKET kernel metadata, with pre- and post-pool standardisation. Uses `src.features.pooling` and `src.features.multirocket`.

In [1]:
# Path fix: use this repo's src
from pathlib import Path
import sys
import numpy as np
PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

import logging
from sklearn.preprocessing import StandardScaler

from src.utils.paths import get_multirocket_features_dir, get_models_dir, get_reduced_dir, ensure_dir
from src.features.pooling import structured_pool, build_dilation_pool_groups
from src.features.memmap import open_memmap_read
from src.features.multirocket import load_multirocket_transformer, extract_kernel_info

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

POOL = "mean"
print("Using PROJECT_ROOT:", PROJECT_ROOT)
print("Pool operator:", POOL)

Using PROJECT_ROOT: C:\Projects\DeepLearningProject
Pool operator: mean


## Kernel-informed dilation-aware pooling

MultiROCKET outputs one value per **(origin × statistic × kernel)**: 2 origins × 4 stats × 2016 kernels = **16,128** features.

The fitted MultiROCKET transformer stores per-kernel metadata including **dilation** (the temporal scale each kernel captures). Kernels sharing the same dilation look at patterns at the same time scale.

**Dilation-aware pooling** groups kernels by their dilation value and pools within each group, separately for each (statistic, origin) pair. Each output feature is one **(dilation, statistic, origin)** triple — a semantically meaningful unit that the FT-Transformer can attend to individually.

With *D* unique dilations the output is *D* × 4 stats × 2 origins features (typically ~192).

**Scaling pipeline** (critical for FT-Transformer's shared `Linear(1, d_token)` projection):
1. **Pre-pool**: `StandardScaler` on the raw 16,128 features (fit on train) so all features contribute equally within each pool group.
2. **Pool**: mean over each (dilation, stat, origin) group.
3. **Post-pool**: `StandardScaler` on the pooled features (fit on train) so every token entering the FT-Transformer has unit variance.

In [2]:
# Load shape metadata from MultiROCKET output
mr_dir = get_multirocket_features_dir()
shapes_path = mr_dir / "shapes.npz"
if not shapes_path.exists():
    raise FileNotFoundError("Run 02_multirocket.ipynb first to create %s" % shapes_path)
shapes = np.load(shapes_path)
train_shape = tuple(shapes["train"])
test_shape = tuple(shapes["test"])
val_shape = tuple(shapes["val"]) if "val" in shapes else None
print("Train shape:", train_shape)
print("Test shape:", test_shape)
if val_shape:
    print("Val shape:", val_shape)

Train shape: (36120, 16128)
Test shape: (6773, 16128)
Val shape: (2257, 16128)


In [3]:
# Load MultiROCKET features (memmap)
print("Loading train.dat...")
F_train = np.array(open_memmap_read(mr_dir / "train.dat", train_shape), dtype=np.float32)
print("Loading test.dat...")
F_test = np.array(open_memmap_read(mr_dir / "test.dat", test_shape), dtype=np.float32)
if val_shape:
    print("Loading val.dat...")
    F_val = np.array(open_memmap_read(mr_dir / "val.dat", val_shape), dtype=np.float32)
else:
    F_val = None
print("F_train %s F_test %s" % (F_train.shape, F_test.shape))

# Load fitted MultiROCKET transformer and extract kernel metadata
mr_transformer_path = get_models_dir() / "multirocket" / "transformer.joblib"
transformer = load_multirocket_transformer(mr_transformer_path)
kinfo = extract_kernel_info(transformer)
print("\nKernel info:")
print("  Dilations (%d levels): %s" % (len(kinfo["dilations"]), kinfo["dilations"]))
print("  Features per dilation: %s" % kinfo["num_features_per_dilation"])
print("  Base features: %d  (84 kernels × sum(nfpd)=%d)" % (kinfo["n_base_features"], kinfo["num_features_per_dilation"].sum()))

Loading train.dat...
Loading test.dat...
Loading val.dat...
F_train (36120, 16128) F_test (6773, 16128)

Kernel info:
  Dilations (22 levels): [  1   2   3   4   5   7   9  12  16  21  28  38  50  66  88 116 154 203
 269 357 472 624]
  Features per dilation: [3 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  Base features: 2016  (84 kernels × sum(nfpd)=24)


In [4]:
# Build dilation-aware pool groups from kernel metadata
group_indices, group_meta = build_dilation_pool_groups(
    dilations=kinfo["dilations"],
    num_features_per_dilation=kinfo["num_features_per_dilation"],
    n_kernels_per_group=kinfo["n_kernels_per_group"],
    n_stats=kinfo["n_stats"],
    n_origins=kinfo["n_origins"],
)
n_groups = len(group_indices)
print("Pool groups: %d  (= %d dilations × %d stats × %d origins)" % (
    n_groups, len(kinfo["dilations"]), kinfo["n_stats"], kinfo["n_origins"]))
print("Group sizes:", sorted(set(m["group_size"] for m in group_meta)))

2026-03-09 16:01:20,080 | INFO | Built 176 dilation-aware pool groups from 22 dilations × 4 stats × 2 origins (input_dim=16128 → output_dim=176)


Pool groups: 176  (= 22 dilations × 4 stats × 2 origins)
Group sizes: [84, 252]


In [5]:
# --- Step 1: Pre-pool standardisation (fit on train only) ---
print("Pre-pool scaling (StandardScaler on %d raw features, fit on train)..." % F_train.shape[1])
pre_scaler = StandardScaler()
F_train_s = pre_scaler.fit_transform(F_train)
F_test_s = pre_scaler.transform(F_test)
F_val_s = pre_scaler.transform(F_val) if F_val is not None else None

# --- Step 2: Dilation-aware pooling ---
print("Pooling (mean over each dilation group)...")
P_train = structured_pool(F_train_s, group_indices=group_indices, pool=POOL)
P_test = structured_pool(F_test_s, group_indices=group_indices, pool=POOL)
P_val = structured_pool(F_val_s, group_indices=group_indices, pool=POOL) if F_val_s is not None else None

# --- Step 3: Post-pool standardisation (fit on train only) ---
print("Post-pool scaling (StandardScaler on %d pooled features, fit on train)..." % P_train.shape[1])
post_scaler = StandardScaler()
P_train = post_scaler.fit_transform(P_train).astype(np.float32)
P_test = post_scaler.transform(P_test).astype(np.float32)
P_val = post_scaler.transform(P_val).astype(np.float32) if P_val is not None else None

# --- Summary ---
print("\nDimension change (dilation-aware pooling + scaling):")
print("  Before: (%d, %d)  ->  After: (%d, %d)" % (F_train.shape[0], F_train.shape[1], P_train.shape[0], P_train.shape[1]))
print("  Reduction: %d -> %d features per sample (factor %.1f)" % (
    F_train.shape[1], P_train.shape[1], F_train.shape[1] / P_train.shape[1]))

Pre-pool scaling (StandardScaler on 16128 raw features, fit on train)...
Pooling (mean over each dilation group)...
Post-pool scaling (StandardScaler on 176 pooled features, fit on train)...

Dimension change (dilation-aware pooling + scaling):
  Before: (36120, 16128)  ->  After: (36120, 176)
  Reduction: 16128 -> 176 features per sample (factor 91.6)


In [6]:
# Save pooled features to data/processed/reduced/pooled/ (used by 05/06 for B1/B2)
out_dir = ensure_dir(get_reduced_dir() / "pooled")
print("Saving to %s..." % out_dir)
np.save(out_dir / "train.npy", P_train)
np.save(out_dir / "test.npy", P_test)
if P_val is not None:
    np.save(out_dir / "val.npy", P_val)

# Persist scalers and group metadata for reproducibility
import joblib
joblib.dump(pre_scaler, out_dir / "pre_scaler.joblib")
joblib.dump(post_scaler, out_dir / "post_scaler.joblib")
np.savez(out_dir / "pool_groups.npz",
         group_sizes=np.array([m["group_size"] for m in group_meta]),
         dilations=np.array([m["dilation"] for m in group_meta]),
         stat_indices=np.array([m["stat_idx"] for m in group_meta]),
         origin_indices=np.array([m["origin_idx"] for m in group_meta]))

print("Done. Saved pooled features (%d dims), scalers, and group metadata." % P_train.shape[1])

Saving to C:\Projects\DeepLearningProject\data\processed\reduced\pooled...
Done. Saved pooled features (176 dims), scalers, and group metadata.
